In [3]:
import os
import json
import glob
import re
import logging
from collections import defaultdict
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
import os
import json

# Define the full set of domains and evaluation datasets
DOMAINS = ["t0", "cot", "flan", "niv"]
EVAL_DATASETS = [
    "orca_t0_val","orca_cot_val","orca_flan_val","orca_niv_val"
]
SIZES = ["one", "half", "third", "double", "triple"]

# Adjust these paths/tokens as needed
ROOT_DIR = "/mbz/users/liyuan/LLaMA-Factory/results/data_mixing/orca"
model_base = "Qwen2.5-1.5B"    # Example model base name
base_token = "660000"           # Example base token
results = []

# ------------------------------
# Main Processing Loop
# ------------------------------
for domain in DOMAINS:
    for size in SIZES:
        # Prepare the “main” dataset for this domain
        dataset1 = f"{base_token}_{domain}_{size}"
        
        # Gather the other five domains with a fixed size of "one"
        # (based on your shell script’s approach)
        other_datasets = []
        for d in DOMAINS:
            if d != domain:
                other_datasets.append(f"{base_token}_{d}_one")
        
        # Accumulate perplexity across all six evaluation datasets
        acc_ppl = 0.0
        valid_file_count = 0

        for eval_dataset in EVAL_DATASETS:
            # -------------------------------------------------------
            # Construct your JSON filename exactly as your shell
            # script produces it. If your shell script's --save_name
            # only uses the first three data tokens, replicate that
            # pattern here. For example:
            #   ppl_{eval_dataset}_{model_base}_{dataset1}_{dataset2}_{dataset3}.json
            # -------------------------------------------------------
            
            # If you truly only have three tokens in the filename:
            # filename = (
            #     f"ppl_{eval_dataset}_{model_base}_{dataset1}_"
            #     f"{other_datasets[0]}_{other_datasets[1]}.json"
            # )
            
            # If your shell script actually includes all five “other” domains,
            # you’d do something like:
            filename = (
                f"ppl_{eval_dataset}_{model_base}_{dataset1}_"
                f"{other_datasets[0]}_{other_datasets[1]}_{other_datasets[2]}.json"
            )
            
            # Build the full path to the JSON file
            filepath = os.path.join(ROOT_DIR, model_base, base_token, domain, filename)
            
            # Check if the file exists
            if not os.path.isfile(filepath):
                print(f"Warning: File not found: {filepath}. Skipping this file.")
                continue
            
            # Load the JSON data
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except json.JSONDecodeError:
                print(f"Error: JSON decoding failed for file: {filepath}. Skipping.")
                continue
            except Exception as e:
                print(f"Error reading {filepath}: {e}. Skipping.")
                continue
            
            # Extract the perplexity from the data structure
            try:
                token_ppl = data[0]["token-level ppl"]
            except (IndexError, KeyError, TypeError):
                print(f"Error: 'token-level ppl' not found or malformed in {filepath}. Skipping.")
                continue
            
            # Accumulate perplexity
            acc_ppl += token_ppl
            valid_file_count += 1

        # Compute the average perplexity if we have valid files
        if valid_file_count == 4:
            avg_ppl = acc_ppl / valid_file_count
        else:
            avg_ppl = None  # or some default indicator

        # Store results
        results.append({
            "domain": domain,
            "size": size,
            "avg_ppl": avg_ppl
        })

# ------------------------------
# Output the Results
# ------------------------------
print(model_base)
for entry in results:
    print(entry)


Qwen2.5-1.5B
{'domain': 't0', 'size': 'one', 'avg_ppl': 2.2866486631430565}
{'domain': 't0', 'size': 'half', 'avg_ppl': 2.2593711727562846}
{'domain': 't0', 'size': 'third', 'avg_ppl': 2.258523739943189}
{'domain': 't0', 'size': 'double', 'avg_ppl': 2.2230923787742363}
{'domain': 't0', 'size': 'triple', 'avg_ppl': 2.206723740483723}
{'domain': 'cot', 'size': 'one', 'avg_ppl': 2.2826284212005445}
{'domain': 'cot', 'size': 'half', 'avg_ppl': 2.26128782978379}
{'domain': 'cot', 'size': 'third', 'avg_ppl': 2.2797535634078034}
{'domain': 'cot', 'size': 'double', 'avg_ppl': 2.204849435686754}
{'domain': 'cot', 'size': 'triple', 'avg_ppl': 2.159635208935335}
{'domain': 'flan', 'size': 'one', 'avg_ppl': 2.2905530230802915}
{'domain': 'flan', 'size': 'half', 'avg_ppl': 2.26311841317225}
{'domain': 'flan', 'size': 'third', 'avg_ppl': 2.2504920973639324}
{'domain': 'flan', 'size': 'double', 'avg_ppl': 2.213505515600158}
{'domain': 'flan', 'size': 'triple', 'avg_ppl': 2.2009228928507687}
{'domain'

In [5]:
from collections import defaultdict

domain_order = DOMAINS
size_order = ['third', 'half', 'one', 'double', 'triple']

# Sort the data based on domain and size order
sorted_data = sorted(
    results,
    key=lambda x: (
        domain_order.index(x['domain']),
        size_order.index(x['size'])
    )
)

grouped_data = defaultdict(list)
for entry in sorted_data:
    grouped_data[entry['domain']].append(entry['avg_ppl'])

# Create the structured n x 5 list
structured_list = [grouped_data[domain] for domain in domain_order]

# Print the structured list
for domain, values in zip(domain_order, structured_list):
    print(f"{domain}: {values}")

t0: [2.258523739943189, 2.2593711727562846, 2.2866486631430565, 2.2230923787742363, 2.206723740483723]
cot: [2.2797535634078034, 2.26128782978379, 2.2826284212005445, 2.204849435686754, 2.159635208935335]
flan: [2.2504920973639324, 2.26311841317225, 2.2905530230802915, 2.213505515600158, 2.2009228928507687]
niv: [2.2540134371180054, 2.261567529221891, 2.25094049401325, 2.2239682767324083, 2.204275983634859]
